In [1]:
# ============================================================
# Notebook: 01_preprocessing_and_intersectional_grouping.ipynb
# Project : Double Exclusion — Intersectional Fairness in VIP Recourse
# Stage   : Data loading, merging, and intersectional group definition
# ============================================================

import pandas as pd
import numpy as np
import zipfile, glob, os

# --- Local paths (edit these on your machine) ---
DATA_DIR = r"C:\Users\miy\Downloads\1000_논문투고\117.금융 합성데이터\3.개방데이터\1.데이터\1. 합성데이터"
WORK_DIR = os.path.join(DATA_DIR, "_extracted")
os.makedirs(WORK_DIR, exist_ok=True)

pd.set_option("display.max_columns", 100)

# Verify the source directory exists without printing the path
assert os.path.isdir(DATA_DIR), "Source directory not found — check DATA_DIR."
print("Setup complete. Source directory located.")

Setup complete. Source directory located.


In [2]:
# Only three sources are required for the VIP-recourse design:
#   01 = member info      -> VIP flag, gender, age, region, credit tenure
#   02 = credit info      -> limits (non-controllable features)
#   03 = approval/sales   -> spending amounts (actionable features)
# CB sheets (09-11) are intentionally excluded to preserve the
# "internal behavioral data only" assumption of the base study.

TARGET_ZIPS = ["01", "02", "03"]

for zpath in glob.glob(os.path.join(DATA_DIR, "*.zip")):
    fname = os.path.basename(zpath)
    if any(fname.startswith(t) or ("." + t) in fname for t in TARGET_ZIPS) \
       or any(t in fname[:4] for t in TARGET_ZIPS):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(WORK_DIR)
        print(f"Extracted: {fname}")

# List what landed in the work directory (filenames only, no full paths)
extracted = [os.path.basename(p) for p in glob.glob(os.path.join(WORK_DIR, "*"))]
print("\nFiles in work directory:")
for f in sorted(extracted):
    print("  ", f)

Extracted: 01.카드 회원정보.zip
Extracted: 02.카드 신용정보.zip
Extracted: 03.카드 승인매출정보.zip

Files in work directory:
   201807_╚╕┐°┴ñ║╕.csv
   201807_╜┬└╬╕┼├Γ┴ñ║╕.csv
   201807_╜┼┐δ┴ñ║╕.csv
   201808_╚╕┐°┴ñ║╕.csv
   201808_╜┬└╬╕┼├Γ┴ñ║╕.csv
   201808_╜┼┐δ┴ñ║╕.csv
   201809_╚╕┐°┴ñ║╕.csv
   201809_╜┬└╬╕┼├Γ┴ñ║╕.csv
   201809_╜┼┐δ┴ñ║╕.csv
   201810_╚╕┐°┴ñ║╕.csv
   201810_╜┬└╬╕┼├Γ┴ñ║╕.csv
   201810_╜┼┐δ┴ñ║╕.csv
   201811_╚╕┐°┴ñ║╕.csv
   201811_╜┬└╬╕┼├Γ┴ñ║╕.csv
   201811_╜┼┐δ┴ñ║╕.csv
   201812_╚╕┐°┴ñ║╕.csv
   201812_╜┬└╬╕┼├Γ┴ñ║╕.csv
   201812_╜┼┐δ┴ñ║╕.csv


In [3]:
csv_paths = sorted(glob.glob(os.path.join(WORK_DIR, "*.csv")))
print(f"Total CSV files: {len(csv_paths)}\n")

ENC = "utf-8-sig"   # confirmed: UTF-8 with BOM, comma-separated

# Signature columns use the *Korean* headers actually present in the files
SIGNATURES = {
    "member":   "VIP등급코드",        # member info
    "approval": "이용금액_신용_R3M",   # approval/sales
    "credit":   "카드이용한도금액",     # credit info
}

def classify(path):
    cols = pd.read_csv(path, nrows=0, encoding=ENC).columns
    for label, sig in SIGNATURES.items():
        if sig in cols:
            return label, len(cols)
    return "unknown", len(cols)

file_index = {}   # label -> list of paths
for p in csv_paths:
    label, ncol = classify(p)
    file_index.setdefault(label, []).append(p)

for label, paths in sorted(file_index.items()):
    ncol = pd.read_csv(paths[0], nrows=0, encoding=ENC).shape[1]
    print(f"{label:10s}: {len(paths)} files, {ncol} columns")

Total CSV files: 18

approval  : 6 files, 430 columns
credit    : 6 files, 42 columns
member    : 6 files, 79 columns


In [4]:
# --- Korean -> English rename maps (only variables this study uses) ---
# Rationale for each block is given so the feature governance is auditable.

MEMBER_COLS = {
    "기준년월": "job_mon",
    "발급회원번호": "member_no",
    "남녀구분코드": "code_gender",        # IMMUTABLE (sensitive)
    "연령": "age",                        # IMMUTABLE (intersection axis)
    "VIP등급코드": "code_vip",            # TARGET source
    "거주시도명": "add_home_si",          # control / IMMUTABLE
    "입회경과개월수_신용": "mn_credit_tenure",  # IMMUTABLE-like (credit history length)
    "Life_Stage": "code_life",           # IMMUTABLE-like
    "회원여부_연체": "yn_delinquent",     # control (excluded from features)
}

APPROVAL_COLS = {
    "발급회원번호": "member_no",
    "이용금액_신용_R3M": "amt_credit_r3m",   # ACTIONABLE (spending, R3M)
    "이용금액_업종기준": "amt_usage_r3m",     # ACTIONABLE
    "이용가맹점수": "cnt_merchant_r3m",       # ACTIONABLE
    "이용금액_쇼핑": "amt_shop",             # ACTIONABLE (category)
    "이용금액_요식": "amt_eat",              # ACTIONABLE
    "이용금액_여유생활": "amt_leisure",       # ACTIONABLE
}

CREDIT_COLS = {
    "발급회원번호": "member_no",
    "카드이용한도금액": "amt_credit_limit",   # NON-CONTROLLABLE (issuer policy)
}

def load_latest(label, colmap):
    """Load only the latest monthly snapshot (201812) and rename to English."""
    # pick the file whose name contains the max yyyymm; fall back to last sorted
    paths = file_index[label]
    latest = max(paths, key=lambda p: os.path.basename(p)[:6])
    df = pd.read_csv(latest, encoding=ENC, usecols=list(colmap.keys()))
    return df.rename(columns=colmap)

member   = load_latest("member",   MEMBER_COLS)
approval = load_latest("approval", APPROVAL_COLS)
credit   = load_latest("credit",   CREDIT_COLS)

print("Loaded shapes:", member.shape, approval.shape, credit.shape)

# --- Merge on member_no (inner: keep members present in all three) ---
df = (member
      .merge(approval.drop(columns=["job_mon"], errors="ignore"), on="member_no", how="inner")
      .merge(credit.drop(columns=["job_mon"], errors="ignore"),   on="member_no", how="inner"))

print("Merged shape:", df.shape)
print("Duplicate member_no:", df["member_no"].duplicated().sum())
df.head(3)

Loaded shapes: (3000000, 9) (3000000, 7) (3000000, 2)
Merged shape: (3000000, 16)
Duplicate member_no: 0


,job_mon,member_no,code_gender,age,code_vip,mn_credit_tenure,yn_delinquent,add_home_si,code_life,amt_credit_r3m,cnt_merchant_r3m,amt_usage_r3m,amt_shop,amt_eat,amt_leisure,amt_credit_limit
0,201812,SYN_0,2,40대,07,71,0,경기,5.자녀성장기(2),4016699,6,595117,141329,0,0,1950765
1,201812,SYN_1,1,30대,_,16,0,서울,4.자녀성장기(1),786755,24,899009,176137,0,0,999717
2,201812,SYN_2,2,40대,_,6,0,충남,4.자녀성장기(1),273763,1,101374,0,0,0,8678806


In [5]:
# --- Target: VIP vs. general ---
# code_vip: "04"-"07" (and "01"-"03" VVIP) = VIP; "_" = general.
# Stored as strings; treat anything not "_" and not null as VIP.
df["code_vip"] = df["code_vip"].astype(str).str.strip()
df["VIP"] = (~df["code_vip"].isin(["_", "nan", ""])).astype(int)

print("Target distribution (VIP):")
print(df["VIP"].value_counts(normalize=True).round(4), "\n")

# --- Inspect the axes before cutting groups ---
print("Age bands:", sorted(df["age"].dropna().unique()))
print("Gender codes:", sorted(df["code_gender"].dropna().unique()))
print("Credit tenure (months) — quartiles:")
print(df["mn_credit_tenure"].describe(percentiles=[.25, .5, .75]).round(1))

Target distribution (VIP):
VIP
0    0.8008
1    0.1992
Name: proportion, dtype: float64 

Age bands: ['20대', '30대', '40대', '50대', '60대', '70대이상']
Gender codes: [1, 2]
Credit tenure (months) — quartiles:
count    3000000.0
mean          75.2
std           72.4
min            6.0
25%           18.0
50%           53.0
75%          105.0
max          337.0
Name: mn_credit_tenure, dtype: float64


In [6]:
# --- Cutoffs (data-driven, not hand-picked) ---
# Credit tenure: short = Q1 or below (<=18m), long = Q3 or above (>=105m)
TENURE_SHORT = df["mn_credit_tenure"].quantile(0.25)   # 18.0
TENURE_LONG  = df["mn_credit_tenure"].quantile(0.75)   # 105.0

MALE, FEMALE = 1, 2
YOUNG_BANDS = ["30대"]
OLD_BANDS   = ["60대", "70대이상"]

def assign_group(row):
    long_hist  = row["mn_credit_tenure"] >= TENURE_LONG
    short_hist = row["mn_credit_tenure"] <= TENURE_SHORT
    if row["age"] in YOUNG_BANDS and row["code_gender"] == MALE and long_hist:
        return "advantaged"      # 30s + male + long credit history
    if row["age"] in OLD_BANDS and row["code_gender"] == FEMALE and short_hist:
        return "disadvantaged"   # 60+ + female + short credit history
    return "other"

df["group"] = df.apply(assign_group, axis=1)

# --- Group sizes and within-group VIP approval rate (the CDR numerator/denominator) ---
summary = (df.groupby("group")
             .agg(n=("VIP", "size"),
                  vip_rate=("VIP", "mean"))
             .reindex(["advantaged", "disadvantaged", "other"]))
summary["vip_rate"] = (summary["vip_rate"] * 100).round(2)
print(f"Cutoffs -> short <= {TENURE_SHORT:.0f}m, long >= {TENURE_LONG:.0f}m\n")
print(summary)

# --- Empirical CDR (raw, pre-model) as a sanity check vs. KCI paper's 75.5x ---
adv = summary.loc["advantaged", "vip_rate"]
dis = summary.loc["disadvantaged", "vip_rate"]
print(f"\nEmpirical approval-rate CDR (raw): {adv/dis:.1f}x"
      if dis > 0 else "\nDisadvantaged VIP rate is 0 — CDR undefined")

Cutoffs -> short <= 18m, long >= 105m

                     n  vip_rate
group                           
advantaged       44315     39.62
disadvantaged    47120      2.05
other          2908565     19.91

Empirical approval-rate CDR (raw): 19.3x


In [7]:
from sklearn.model_selection import train_test_split

IMMUTABLE = ["code_gender", "age", "add_home_si", "mn_credit_tenure", "code_life"]
ACTIONABLE = ["amt_credit_r3m", "amt_usage_r3m", "cnt_merchant_r3m",
              "amt_shop", "amt_eat", "amt_leisure"]
NON_CONTROLLABLE = ["amt_credit_limit"]
FEATURES = IMMUTABLE + ACTIONABLE + NON_CONTROLLABLE
TARGET = "VIP"

model_df = df[FEATURES + [TARGET, "group"]].copy()

# --- Encode categoricals ---
AGE_ORDER = ["20대", "30대", "40대", "50대", "60대", "70대이상"]
model_df["age"] = pd.Categorical(model_df["age"], categories=AGE_ORDER, ordered=True).codes
model_df["code_gender"] = model_df["code_gender"].astype("category").cat.codes
model_df["add_home_si"] = model_df["add_home_si"].astype("category").cat.codes
model_df["code_life"] = model_df["code_life"].astype(str).astype("category").cat.codes

# --- Stratified large sample (preserve VIP ratio AND group composition) ---
SAMPLE_N = 200000   # large enough for power, small enough to be fast
strat_key = model_df["group"].astype(str) + "_" + model_df[TARGET].astype(str)

sample_df, _ = train_test_split(
    model_df, train_size=SAMPLE_N, stratify=strat_key, random_state=42)

print(f"Sampled {len(sample_df):,} of {len(model_df):,} rows")
print("VIP rate — full: {:.4f}  sample: {:.4f}".format(
    model_df[TARGET].mean(), sample_df[TARGET].mean()))
print("\nGroup composition in sample:")
print(sample_df["group"].value_counts())

# --- Train/test split on the stratified sample ---
X = sample_df[FEATURES]
y = sample_df[TARGET]
strat_key2 = sample_df["group"].astype(str) + "_" + y.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=strat_key2, random_state=42)

# Keep group labels aligned with the test set for later group-wise analysis
group_test = sample_df.loc[X_test.index, "group"]

print("\nTrain:", X_train.shape, " Test:", X_test.shape)
print("Train VIP rate:", round(y_train.mean(), 4),
      " Test VIP rate:", round(y_test.mean(), 4))

Sampled 200,000 of 3,000,000 rows
VIP rate — full: 0.1992  sample: 0.1992

Group composition in sample:
group
other            193905
disadvantaged      3141
advantaged         2954
Name: count, dtype: int64

Train: (150000, 12)  Test: (50000, 12)
Train VIP rate: 0.1992  Test VIP rate: 0.1992


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, roc_auc_score, classification_report

# class_weight='balanced' mirrors the base study's handling of 1:4 imbalance
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("F1     :", round(f1_score(y_test, y_pred), 4))
print("Recall :", round(recall_score(y_test, y_pred), 4))
print("AUC-ROC:", round(roc_auc_score(y_test, y_proba), 4))
print("\n", classification_report(y_test, y_pred, digits=4))

F1     : 0.664
Recall : 0.5716
AUC-ROC: 0.8915

               precision    recall  f1-score   support

           0     0.9003    0.9627    0.9305     40040
           1     0.7921    0.5716    0.6640      9960

    accuracy                         0.8848     50000
   macro avg     0.8462    0.7671    0.7972     50000
weighted avg     0.8788    0.8848    0.8774     50000



In [9]:
import numpy as np, pandas as pd

# 0) Sanity: is rf trained on the 200k stratified sample?
print("Test set size:", len(X_test))
print("Groups in test set:")
print(group_test.value_counts(), "\n")   # group_test came from Cell 7

# 1) Global importance via RF's built-in impurity importance (instant)
imp = (pd.DataFrame({"feature": FEATURES,
                     "importance": rf.feature_importances_})
       .sort_values("importance", ascending=False)
       .reset_index(drop=True))
print("RF feature importance ranking:")
print(imp.to_string(index=False), "\n")

# 2) Gender direction: predicted VIP prob by gender, holding the sample as-is
#    (mirrors the KCI finding that female associates with lower VIP odds)
tmp = X_test.copy()
tmp["proba"] = rf.predict_proba(X_test[FEATURES])[:, 1]
print("Mean predicted VIP probability by gender code:")
print(tmp.groupby("code_gender")["proba"].mean().round(4))

Test set size: 50000
Groups in test set:
group
other            48476
disadvantaged      785
advantaged         739
Name: count, dtype: int64 

RF feature importance ranking:
         feature  importance
  amt_credit_r3m    0.261949
   amt_usage_r3m    0.159520
mn_credit_tenure    0.132235
amt_credit_limit    0.118820
        amt_shop    0.101892
cnt_merchant_r3m    0.081907
     add_home_si    0.039645
         amt_eat    0.037308
       code_life    0.024311
             age    0.019501
     amt_leisure    0.013199
     code_gender    0.009713 

Mean predicted VIP probability by gender code:
code_gender
0    0.2151
1    0.1728
Name: proba, dtype: float64


In [10]:
# Score the test set and compare predicted VIP probability across groups
proba_test = rf.predict_proba(X_test[FEATURES])[:, 1]
cdr_tbl = (pd.DataFrame({"group": group_test.values, "proba": proba_test})
             .groupby("group")["proba"].mean()
             .reindex(["advantaged", "disadvantaged", "other"]) * 100).round(3)

print("Mean predicted VIP probability by group (%):")
print(cdr_tbl, "\n")

cdr_model = cdr_tbl["advantaged"] / cdr_tbl["disadvantaged"]
print(f"Model-based CDR (advantaged / disadvantaged): {cdr_model:.1f}x")
print(f"  raw pre-model CDR : 19.3x")
print(f"  KCI paper reported: 75.5x")

Mean predicted VIP probability by group (%):
group
advantaged       36.775
disadvantaged     2.129
other            19.531
Name: proba, dtype: float64 

Model-based CDR (advantaged / disadvantaged): 17.3x
  raw pre-model CDR : 19.3x
  KCI paper reported: 75.5x


In [11]:
# pip install dice-ml   (run once if not installed)
import dice_ml
from dice_ml import Dice

# --- Who gets recourse? Members the model REJECTED (predicted VIP=0) ---
# We focus on advantaged vs. disadvantaged, per the study design.
test_df = X_test.copy()
test_df["group"] = group_test.values
test_df["pred"]  = rf.predict(X_test[FEATURES])

rejected = test_df[(test_df["pred"] == 0) &
                   (test_df["group"].isin(["advantaged", "disadvantaged"]))].copy()

print("Rejected members eligible for recourse:")
print(rejected["group"].value_counts(), "\n")

# --- DiCE data & model interfaces ---
# Continuous vs. categorical must be declared for DiCE's search.
CONTINUOUS = ["mn_credit_tenure", "amt_credit_r3m", "amt_usage_r3m",
              "cnt_merchant_r3m", "amt_shop", "amt_eat", "amt_leisure",
              "amt_credit_limit", "age"]   # age is encoded ordinal here
CATEGORICAL = ["code_gender", "add_home_si", "code_life"]

train_for_dice = X_train.copy()
train_for_dice[TARGET] = y_train.values

d = dice_ml.Data(dataframe=train_for_dice,
                 continuous_features=CONTINUOUS,
                 outcome_name=TARGET)
m = dice_ml.Model(model=rf, backend="sklearn")
exp = Dice(d, m, method="random")

print("DiCE ready. Continuous:", len(CONTINUOUS), " Categorical:", len(CATEGORICAL))

Rejected members eligible for recourse:
group
disadvantaged    785
advantaged       518
Name: count, dtype: int64 

DiCE ready. Continuous: 9  Categorical: 3


In [12]:
import time, logging
# Silence DiCE's per-row tqdm/logging noise
logging.getLogger("dice_ml").setLevel(logging.ERROR)
try:
    import tqdm
    from functools import partialmethod
    tqdm.tqdm.__init__ = partialmethod(tqdm.tqdm.__init__, disable=True)
except Exception:
    pass

RECOURSE_N = 200   # per-group sample for recourse (stratified from rejected)

def take_sample(sub):
    n = min(RECOURSE_N, len(sub))
    return sub.sample(n=n, random_state=42)

ACTION_HORIZON = 0.20
vary_A = FEATURES
vary_B = [f for f in FEATURES if f not in IMMUTABLE]
vary_C = ACTIONABLE

def per_feature_ranges(row):
    rng = {}
    for f in ACTIONABLE:
        v = float(row[f]); lo, hi = sorted([v*(1-ACTION_HORIZON), v*(1+ACTION_HORIZON)])
        rng[f] = [lo, hi]
    return rng

def gen_recourse(subset, vary, use_range, tag):
    results, t0 = [], time.time()
    for i, (_, row) in enumerate(subset.iterrows()):
        query = row[FEATURES].to_frame().T
        try:
            kw = dict(total_CFs=4, desired_class=1, features_to_vary=vary,
                      verbose=False)
            if use_range:
                kw["permitted_range"] = per_feature_ranges(row)
            cf = exp.generate_counterfactuals(query, **kw)
            cfdf = cf.cf_examples_list[0].final_cfs_df
            results.append(None if cfdf is None else cfdf[FEATURES].reset_index(drop=True))
        except Exception:
            results.append(None)
    print(f"  [{tag}] {len(subset)} done in {time.time()-t0:.0f}s")
    return results

recourse = {}
for gname, sub in [("disadvantaged", rejected[rejected.group=="disadvantaged"]),
                   ("advantaged",    rejected[rejected.group=="advantaged"])]:
    s = take_sample(sub)
    print(f"\n=== {gname} (using n={len(s)} of {len(sub)}) ===")
    recourse[gname] = {
        "A": gen_recourse(s, vary_A, False, f"{gname}-A"),
        "B": gen_recourse(s, vary_B, False, f"{gname}-B"),
        "C": gen_recourse(s, vary_C, True,  f"{gname}-C"),
    }
    recourse[gname]["_sample"] = s   # keep the queried rows for violation checks
print("\nDone.")


=== disadvantaged (using n=200 of 785) ===
  [disadvantaged-A] 200 done in 127s
  [disadvantaged-B] 200 done in 137s
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time 

In [13]:
import numpy as np, pandas as pd

ACTION_HORIZON = 0.20

def check_path(orig_row, cf_row):
    """Return violation flags for one counterfactual path."""
    # VR: any immutable feature changed?
    imm_viol = any(not np.isclose(float(orig_row[f]), float(cf_row[f]))
                   for f in IMMUTABLE)
    # Action VR: any actionable feature moved beyond ±20%?
    act_viol = False
    for f in ACTIONABLE:
        o = float(orig_row[f]); c = float(cf_row[f])
        if o == 0:
            act_viol |= (c != 0)
        else:
            act_viol |= abs(c - o) / abs(o) > ACTION_HORIZON + 1e-9
    return imm_viol, act_viol

def score_scenario(sample_df, cf_list):
    """Per-borrower: found any path? reliable path? Aggregate to rates."""
    n = len(sample_df)
    rr_hits = 0            # borrower got >=1 valid CF (success, any quality)
    reliable_borrowers = 0 # borrower got >=1 fully-reliable CF
    path_total = 0
    path_reliable = 0
    for (_, orig), cfs in zip(sample_df.iterrows(), cf_list):
        if cfs is None or len(cfs) == 0:
            continue
        rr_hits += 1
        borrower_reliable = False
        for _, cf in cfs.iterrows():
            path_total += 1
            imm_v, act_v = check_path(orig, cf)
            if not imm_v and not act_v:
                path_reliable += 1
                borrower_reliable = True
        if borrower_reliable:
            reliable_borrowers += 1
    return {
        "RR": rr_hits / n,
        "Borrower_Reliable_RR": reliable_borrowers / n,
        "Pathway_Reliable_RR": (path_reliable / path_total) if path_total else 0.0,
        "n": n,
    }

rows = []
for gname in ["advantaged", "disadvantaged"]:
    s = recourse[gname]["_sample"]
    for scen in ["A", "B", "C"]:
        r = score_scenario(s, recourse[gname][scen])
        r.update({"group": gname, "scenario": scen})
        rows.append(r)

res = pd.DataFrame(rows)[["group", "scenario", "n", "RR",
                          "Pathway_Reliable_RR", "Borrower_Reliable_RR"]]
for c in ["RR", "Pathway_Reliable_RR", "Borrower_Reliable_RR"]:
    res[c] = (res[c] * 100).round(2)
print(res.to_string(index=False))

        group scenario   n    RR  Pathway_Reliable_RR  Borrower_Reliable_RR
   advantaged        A 200 100.0                 2.00                   4.5
   advantaged        B 200 100.0                 2.38                   6.0
   advantaged        C 200  16.0                97.41                  16.0
disadvantaged        A 200 100.0                 0.00                   0.0
disadvantaged        B 200 100.0                 0.00                   0.0
disadvantaged        C 200   0.0                 0.00                   0.0


In [14]:
import pandas as pd

# --- Axis 1: rejection probability (from model-based group VIP probability) ---
# Using the group mean predicted VIP probability computed earlier (Cell 10).
p_vip = {"advantaged": 0.36775, "disadvantaged": 0.02129}   # from CDR table

# --- Axis 2: probability that recourse is NOT reliable ---
# Use scenario C (the deployable guardrail) Borrower_Reliable_RR.
rel_rr_C = (res[res.scenario == "C"]
            .set_index("group")["Borrower_Reliable_RR"] / 100).to_dict()

print("Double Exclusion Index (Method A: conditional-probability combination)\n")
dei_rows = []
for g in ["advantaged", "disadvantaged"]:
    p_reject   = 1 - p_vip[g]                    # P(not approved)
    p_no_recourse = 1 - rel_rr_C.get(g, 0.0)     # P(no reliable recourse | rejected)
    dei = p_reject * p_no_recourse               # joint exclusion probability
    dei_rows.append({"group": g,
                     "P_reject": round(p_reject, 4),
                     "P_no_reliable_recourse": round(p_no_recourse, 4),
                     "DEI": round(dei, 4)})
dei_df = pd.DataFrame(dei_rows)
print(dei_df.to_string(index=False))

ratio = dei_df.set_index("group").loc["disadvantaged", "DEI"] / \
        dei_df.set_index("group").loc["advantaged", "DEI"]
print(f"\nDEI ratio (disadvantaged / advantaged): {ratio:.2f}x")
print("\nInterpretation: probability a member is BOTH rejected AND left "
      "without any reliable recourse path.")

Double Exclusion Index (Method A: conditional-probability combination)

        group  P_reject  P_no_reliable_recourse    DEI
   advantaged    0.6322                    0.84 0.5311
disadvantaged    0.9787                    1.00 0.9787

DEI ratio (disadvantaged / advantaged): 1.84x

Interpretation: probability a member is BOTH rejected AND left without any reliable recourse path.


In [15]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
N_BOOT = 2000   # bootstrap resamples

def borrower_flags(sample_df, cf_list):
    """Per-borrower binary flags: (found_any, reliable_any). Length = n."""
    found, reliable = [], []
    for (_, orig), cfs in zip(sample_df.iterrows(), cf_list):
        if cfs is None or len(cfs) == 0:
            found.append(0); reliable.append(0); continue
        found.append(1)
        rel = False
        for _, cf in cfs.iterrows():
            imm_v, act_v = check_path(orig, cf)   # from Cell 13
            if not imm_v and not act_v:
                rel = True; break
        reliable.append(1 if rel else 0)
    return np.array(found), np.array(reliable)

def boot_ci(flags, n_boot=N_BOOT):
    """Percentile 95% CI of the mean of a 0/1 array."""
    n = len(flags)
    means = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)      # resample with replacement
        means[b] = flags[idx].mean()
    lo, hi = np.percentile(means, [2.5, 97.5])
    return flags.mean(), lo, hi

ci_rows = []
for gname in ["advantaged", "disadvantaged"]:
    s = recourse[gname]["_sample"]
    for scen in ["A", "B", "C"]:
        found, reliable = borrower_flags(s, recourse[gname][scen])
        for metric_name, arr in [("RR", found),
                                 ("Borrower_Reliable_RR", reliable)]:
            m, lo, hi = boot_ci(arr)
            ci_rows.append({
                "group": gname, "scenario": scen, "metric": metric_name,
                "estimate_%": round(m*100, 2),
                "CI95_low_%": round(lo*100, 2),
                "CI95_high_%": round(hi*100, 2),
            })

ci_df = pd.DataFrame(ci_rows)
print(ci_df.to_string(index=False))

        group scenario               metric  estimate_%  CI95_low_%  CI95_high_%
   advantaged        A                   RR       100.0       100.0        100.0
   advantaged        A Borrower_Reliable_RR         4.5         2.0          7.5
   advantaged        B                   RR       100.0       100.0        100.0
   advantaged        B Borrower_Reliable_RR         6.0         3.0          9.5
   advantaged        C                   RR        16.0        11.0         21.0
   advantaged        C Borrower_Reliable_RR        16.0        11.0         21.5
disadvantaged        A                   RR       100.0       100.0        100.0
disadvantaged        A Borrower_Reliable_RR         0.0         0.0          0.0
disadvantaged        B                   RR       100.0       100.0        100.0
disadvantaged        B Borrower_Reliable_RR         0.0         0.0          0.0
disadvantaged        C                   RR         0.0         0.0          0.0
disadvantaged        C Borro

In [16]:
import numpy as np
from scipy.stats import beta

def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (0.0, 0.0, 1.0)
    z = 1.959963985
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return p, max(0, center - half), min(1, center + half)

def rule_of_three(n):
    """Upper 95% bound when 0 events observed: ~3/n."""
    return 3.0 / n if n > 0 else 1.0

print("Exact binomial bounds for the zero-count disadvantaged cells\n")

# (a) On the 200-member recourse sample
n_sample = 200
for scen in ["A", "B", "C"]:
    p, lo, hi = wilson_ci(0, n_sample)
    print(f"[sample n=200] disadvantaged-{scen} Reliable RR: "
          f"{p*100:.2f}%  Wilson95 upper = {hi*100:.2f}%  "
          f"(Rule-of-3 upper = {rule_of_three(n_sample)*100:.2f}%)")

# (b) Extrapolated bound if the same 0-rate held over the FULL rejected pool
n_full_dis = int((rejected["group"] == "disadvantaged").sum())
p, lo, hi = wilson_ci(0, n_full_dis)
print(f"\n[full rejected pool n={n_full_dis}] disadvantaged Reliable RR "
      f"upper bound (Wilson95) = {hi*100:.3f}%  "
      f"(Rule-of-3 = {rule_of_three(n_full_dis)*100:.3f}%)")

# Contrast: advantaged scenario C with exact CI on the same sample
k_adv = int(round(0.16 * 200))
p, lo, hi = wilson_ci(k_adv, 200)
print(f"\n[reference] advantaged-C Reliable RR: {p*100:.1f}%  "
      f"Wilson95 = [{lo*100:.1f}%, {hi*100:.1f}%]")

Exact binomial bounds for the zero-count disadvantaged cells

[sample n=200] disadvantaged-A Reliable RR: 0.00%  Wilson95 upper = 1.88%  (Rule-of-3 upper = 1.50%)
[sample n=200] disadvantaged-B Reliable RR: 0.00%  Wilson95 upper = 1.88%  (Rule-of-3 upper = 1.50%)
[sample n=200] disadvantaged-C Reliable RR: 0.00%  Wilson95 upper = 1.88%  (Rule-of-3 upper = 1.50%)

[full rejected pool n=785] disadvantaged Reliable RR upper bound (Wilson95) = 0.487%  (Rule-of-3 = 0.382%)

[reference] advantaged-C Reliable RR: 16.0%  Wilson95 = [11.6%, 21.7%]


In [17]:
import time, pickle, os

dis_full = rejected[rejected.group == "disadvantaged"].copy()
print(f"Disadvantaged rejected pool: n={len(dis_full)}\n")

CKPT_RECOURSE = os.path.join(WORK_DIR, "recourse_dis_full.pkl")

# Resume if a checkpoint exists (kernel-restart safe)
if os.path.exists(CKPT_RECOURSE):
    with open(CKPT_RECOURSE, "rb") as f:
        recourse_dis_full = pickle.load(f)
    print("Loaded existing checkpoint.")
else:
    recourse_dis_full = {}

for scen, (vary, use_range) in {
        "A": (vary_A, False),
        "B": (vary_B, False),
        "C": (vary_C, True)}.items():
    if scen in recourse_dis_full:
        print(f"[{scen}] already done, skipping.")
        continue
    t0 = time.time()
    recourse_dis_full[scen] = gen_recourse(dis_full, vary, use_range, f"dis-full-{scen}")
    with open(CKPT_RECOURSE, "wb") as f:
        pickle.dump(recourse_dis_full, f)   # save after each scenario
    print(f"  [{scen}] saved. elapsed {time.time()-t0:.0f}s")

print("\nFull-population recourse complete.")

Disadvantaged rejected pool: n=785

  [dis-full-A] 785 done in 800s
  [A] saved. elapsed 800s
  [dis-full-B] 785 done in 2740s
  [B] saved. elapsed 2740s
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
No Counterfactuals found for the given configuration, perhaps try with d

In [18]:
import numpy as np, pandas as pd
from scipy.stats import beta

# Reuse borrower_flags / check_path from earlier cells.
dis_full = rejected[rejected.group == "disadvantaged"].copy()

def clopper_pearson_upper(k, n, alpha=0.05):
    """Exact one-sided upper 95% bound (handles k=0 correctly)."""
    if k == n:
        return 1.0
    return beta.ppf(1 - alpha/2, k + 1, n - k)

print(f"Full disadvantaged rejected pool: n={len(dis_full)}\n")
rows = []
for scen in ["A", "B", "C"]:
    found, reliable = borrower_flags(dis_full, recourse_dis_full[scen])
    n = len(dis_full)
    k_found = int(found.sum())
    k_rel   = int(reliable.sum())
    rows.append({
        "scenario": scen,
        "n": n,
        "RR_%": round(100*k_found/n, 3),
        "n_reliable": k_rel,
        "Reliable_RR_%": round(100*k_rel/n, 3),
        "Reliable_upper95_%": round(100*clopper_pearson_upper(k_rel, n), 3),
    })

full_res = pd.DataFrame(rows)
print(full_res.to_string(index=False))

Full disadvantaged rejected pool: n=785

scenario   n    RR_%  n_reliable  Reliable_RR_%  Reliable_upper95_%
       A 785 100.000           1          0.127               0.708
       B 785 100.000           2          0.255               0.917
       C 785   0.127           1          0.127               0.708
